In [ ]:
import inspect
import os
import subprocess
import sys
import time

from pathlib import Path
from tqdm import tqdm


from lhotse import (
    CutSet,
    load_manifest_lazy
)

from transformers import (
    AutoProcessor,
    AutoModelForCTC,
    Wav2Vec2Processor, 
    Wav2Vec2ForCTC 
)

#if you need to mount your code entry point
sys.path.append("native-L2-coordinate-geometry/src")


from config import (
    DATA_PATH,
    MANIFEST_PATH,
    SANDI_CEFR_DICT,
    MODELS,
)

from build_phone_codes import (
    AveragingStrategy,
    build_average_phone_matrix_for_timit,
    build_average_phone_matrix_from_tuple_alignments,
    build_phone_averages_from_pickles,
    
    
)

from utils import (
    get_cuts,
    load_lexicon,
    load_pickle,
    PhoneClass,
    save_pickle,
)

#############
#Variables
#############
#label for phone assessment
PHONE_CLASS = PhoneClass.DIPHONE
AVG_STRATEGY = AveragingStrategy.CENTER_PHONE

DEVICE='cuda'
FRAME_SHIFT = 0.02 
RANK = 16


#############
#Data
#############
#examples: either call get_cuts or use lhotse from_file

DATA_PARTS_TIMIT = ['TRAIN', 'DEV', 'TEST']
cutsets_timit = get_cuts(corpus_name='timit', partition_names=DATA_PARTS_TIMIT, manifest_dir=MANIFEST_PATH)
cuts_train_timit = cutsets_timit['TRAIN']
cuts_dev_timit = cutsets_timit['DEV']
cuts_test_timit = cutsets_timit['TEST']


sandi_cutset_path =  /path/to/sandi
cuts_train_sandi = CutSet.from_file(
    sandi_cutset_path
)


##############
#Alignments
##############
alignment_paths = {
    "factored_hybrid_diphone": {
        "train": DATA_PATH / "alignments/fh/sandi.train.forced.pickle",
        "dev": DATA_PATH / "alignments/fh/sandi.dev.forced.pickle",
    },
    "wav2vec_timitv3": {
        "train": DATA_PATH / "alignments/ctc/sandi.train.forced.pickle",
        "dev": DATA_PATH / "alignments/ctc/sandi.dev.forced.pickle",
    },
}

selected_alignment = "factored_hybrid_diphone"
selected_splits = ["train", "dev"]

alignments = {
    selected_alignment: {
        split: load_pickle(alignment_paths[selected_alignment][split])
        for split in selected_splits
    }
}

train_fh_algn_sandi = alignments[selected_alignment]['train']
dev_fh_algn_sandi   = alignments[selected_alignment]['dev']



